In [1]:
# --- STEP 1: INSTALL THE SPEED STACK ---
# We install Unsloth (specifically for Colab T4)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-2y5zwu7g/unsloth_958f3741394f4e0aab3966fe006c494c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-2y5zwu7g/unsloth_958f3741394f4e0aab3966fe006c494c
  Resolved https://github.com/unslothai/unsloth.git to commit b036191859194bf637a19ec3b7aaeee75f89f051
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.5/376.5 kB 42.3 MB/s eta 0:00:

In [2]:
%pwd

'/content'

In [3]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
# --- STEP 2: LOAD THE MODEL (Llama-3-8B) ---
# We use 8B because it is SMARTER than 3B at JSON, and Unsloth fits it on T4 easily.
max_seq_length = 2048
dtype = None # None = auto detection
load_in_4bit = True

print("Loading Llama-3 Model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit", # The 4-bit quantized version
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

Loading Llama-3 Model...
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

In [5]:
# --- STEP 3: CONFIGURE LoRA (The "Adapter") ---
# We add LoRA adapters which are the only parts we actually train
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized to 0
    bias = "none",    # Optimized to none
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth 2026.2.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [6]:
# --- STEP 4: LOAD YOUR DATA ---
# We defined your prompt format in the data gen script.
# Here we just map it to what the model expects.

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

print("Loading your custom dataset...")
dataset = load_dataset("json", data_files="medical_instruct_train.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True,)


Loading your custom dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [7]:
# --- STEP 5: TRAIN (The Main Event) ---
print("Starting Training (this will take ~15-20 mins)...")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can speed up training for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # We only do 60 steps for the "Resume Speedrun" (enough to learn JSON)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

Starting Training (this will take ~15-20 mins)...


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [8]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: akhilgalla (foobar41) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss
1,2.423700
2,2.074800
3,1.945900
4,1.752900
5,1.709500
6,1.488100
7,1.241400
8,1.273000
9,0.970300
10,0.873500


train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
train/grad_norm,█▆▆▅▆▆▂▄▄▄▆█▅▃▂▁▃▂▂▁▂▁▁▁▁▁▂▁▁▁▁▁▁▂▁▁▂▁▁▁
train/learning_rate,▂▄▅▇██▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁
train/loss,█▇▆▆▅▄▃▂▂▂▂▂▂▂▁▂▂▁▁▁▂▁▁▁▂▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁
total_flos,4308687901605888.0
train/epoch,0.96
train/global_step,60
train/grad_norm,0.35942
train/learning_rate,0.0
train/loss,0.5745


TrainOutput(global_step=60, training_loss=0.7713236813743909, metrics={'train_runtime': 379.6942, 'train_samples_per_second': 1.264, 'train_steps_per_second': 0.158, 'total_flos': 4308687901605888.0, 'train_loss': 0.7713236813743909, 'epoch': 0.96})

In [9]:
# --- STEP 6: TEST IT (Inference) ---
print("\nRunning Inference Test...")
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

inputs = tokenizer(
    [
        alpaca_prompt.format(
            "You are a medical diagnostic assistant. Output a valid JSON response.", # Instruction
            "Patient complains of sharp chest pain worsening with deep breaths.", # Input
            "", # Output - leave this blank for generation!
        )
    ], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
print(tokenizer.batch_decode(outputs))


Running Inference Test...
['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nYou are a medical diagnostic assistant. Output a valid JSON response.\n\n### Input:\nPatient complains of sharp chest pain worsening with deep breaths.\n\n### Response:\n{"topic": "Chest Pain", "answer": "Sharp chest pain that worsens with deep breaths is a common symptom of pneumothorax. Pneumothorax is a medical condition in which air or gas enters the space between the lungs and the chest wall, causing the lung to collapse. This can occur as a result of trauma, a lung injury, or other medical conditions. Sharp chest pain that worsens with deep breaths is a classic symptom of pneumothorax, and it is often accompanied by other symptoms such as shortness of breath, coughing, and difficulty breathing. Treatment for pneumothor']


In [10]:
# --- STEP 7: SAVE ---
# This saves the LoRA adapters to your Colab folder
model.save_pretrained("lora_model")
print("\nDONE! Model saved to 'lora_model' folder.")


DONE! Model saved to 'lora_model' folder.


In [11]:
tokenizer.save_pretrained("lora_model")

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/chat_template.jinja',
 'lora_model/tokenizer.json')